In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import os
import h5py
# os.environ['PROJ_LIB'] = '/home/lty/anaconda3/envs/hloc/share/proj'

from osgeo import gdal, osr
from geopy.distance import geodesic

# from scipy.stats import pairs

from hloc import extract_features, match_features
from hloc.utils.io import get_matches, get_keypoints
from hloc.visualization import plot_images, plot_keypoints, plot_matches, read_image, add_text

from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle


In [2]:
image_dir = Path("/home/lty/datasets_my/DJI/m300/")
seu_uav_dir = image_dir / "seu_uav_0103_2"
seu_tif_dir = image_dir / "seu_tif_m300"
output_dir = Path("/home/lty/outputs/scene_match_0103_seu_2")
output_dir.mkdir(exist_ok=True, parents= True)
pairs_path = output_dir/"pairs_m300.txt"
img_list_path = output_dir/"img_list.txt"
uav_list_path = output_dir/"uav_list.txt"
tif_list_path = output_dir/"tif_list.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"
loc_path = output_dir/"loc.txt"# 保存定位结果

In [46]:
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff']
# 收集 'seu_uav' 文件夹中的所有图像文件
uav_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_uav_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

# 收集 'seu_tif' 文件夹中的所有图像文件
tif_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_tif_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]
uav_images = sorted(uav_images, key = sort_key)
tif_images = sorted(tif_images, key = sort_key)
# 将 UAV 和 TIF 图像文件列表合并
img_list = uav_images + tif_images
# 打印 img_list 以验证结果
# for img in img_list:
#     print(img)
#save img_list
with open(img_list_path, 'w') as f:
    for img in img_list:
        f.write(img + "\n")

with open(uav_list_path, 'w') as f:
    for img in uav_images:
        f.write(img + "\n")
with open(tif_list_path, 'w') as f:
    for img in tif_images:
        f.write(img + "\n")

In [47]:
import time
t0 = time.time()
extract_features.main(
    conf=extract_features.confs['superpoint_max'],
    image_dir=image_dir,
    image_list=img_list,
    feature_path = features_path,
)
t1 = time.time()
match_features.main(
    conf=match_features.confs["superpoint+lightglue"],
    pairs=pairs_path,
    features=features_path,
    matches=matches_path,
)
t2 = time.time()
print(f"Feature extraction time: {t1-t0:.3f}s")
print(f"Feature matching time: {t2-t1:.3f}s")

[2025/01/09 15:45:25 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}


Loaded SuperPoint model


100%|██████████| 1213/1213 [01:08<00:00, 17.58it/s]
[2025/01/09 15:46:34 hloc INFO] Finished exporting features.
[2025/01/09 15:46:34 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
100%|██████████| 1178/1178 [00:45<00:00, 26.13it/s]
[2025/01/09 15:47:19 hloc INFO] Finished exporting matches.


Feature extraction time: 69.050s
Feature matching time: 45.234s


In [3]:
geotransform = []
# 打开 geotransform.txt 文件
with open("/home/lty/data/seu_geotransform_m300.txt", "r") as f:
    # 逐行读取文件内容
    for line in f:
        # 去除行首尾的空白字符和换行符
        line = line.strip()
        if line:
            try:
                # 将字符串转换为浮点数
                value = float(line)
                # 将数值添加到列表中
                geotransform.append(value)
            except ValueError:
                print(f"无法将以下内容转换为数值：'{line}'")
                # 根据需要，可以选择跳过或停止程序
                continue
# 输出读取到的 geotransform 数据
print("Geotransform 数组：\n", geotransform)

Geotransform 数组：
 [668601.89603705, 0.03459999999999788, 0.0, 3548451.1491134795, 0.0, -0.03459999999998963]


In [49]:

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0
with open(loc_path, 'w') as loc_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC)
        print(H)
        if H is not None:
            h_uav, w_uav = 1080, 1920
            center_uav = np.array([[w_uav / 2, h_uav / 2]], dtype=np.float32)  
            center_uav[0][1] = center_uav[0][1]-80.0# 形状为 (1, 2)
            # 将中心点坐标转换为齐次坐标
            center_uav_homogeneous = np.array([center_uav[0][0], center_uav[0][1], 1.0])  # 形状为 (3,)
            # 通过单应性矩阵进行变换
            center_tif_homogeneous = np.dot(H, center_uav_homogeneous)  # 形状为 (3,)
            # 归一化
            center_tif = center_tif_homogeneous[:2] / center_tif_homogeneous[2]  # 形状为 (2,)
            angle = extract_rotation_angle(H)
            print(f"旋转角度 (度): {angle}")
            # 打印结果
            print(f"无人机图像中心点在tif的位置：{center_tif}")
            pixel_x, pixel_y = center_tif
            tif_name = os.path.basename(img_tif)
            match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
            if match:
                start_x = match.group(2)
                start_y = match.group(3)
                x_in_map = int(start_x) + pixel_x
                y_in_map = int(start_y) + pixel_y
                print(f"无人机图像中心点在地图上的位置：{x_in_map},{y_in_map}")
            try:
                lon, lat, x_geo, y_geo = pixel_to_geo_coordinates(x_in_map, y_in_map, geotransform)
                print(f"无人机图像中心点的经纬度：{lat}, {lon}")
                if n == 0:
                    x_origin, y_origin = x_geo, y_geo
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {x_geo:.8f} {y_geo:.8f} {x_geo-x_origin:.10f} {y_origin-y_geo:.10f} {angle:.8f}\n")
                n+=1
            except Exception as e:
                print(f"无法计算无人机图像中心点的经纬度：{e}")
end = time.time()
print(f"Time: {end-start:.3f}s")

Found 1178 image pairs.
UAV: seu_uav_0103_2/00000.png - TIF: seu_tif_m300/22_0_4500.tif
(1405, 2)
[[ 1.35491909e+00 -1.19164930e-01 -6.82563489e+02]
 [ 7.80818557e-03  1.30243466e+00  8.82863590e+02]
 [-4.31054471e-05 -5.64909729e-05  1.00000000e+00]]
旋转角度 (度): 2.735614191377284
无人机图像中心点在tif的位置：[ 604.03504635 1597.0692826 ]
无人机图像中心点在地图上的位置：604.0350463543263,6097.069282600032
无人机图像中心点的经纬度：32.05782160769662, 118.78620644324768
UAV: seu_uav_0103_2/00073.png - TIF: seu_tif_m300/22_0_4500.tif
(1371, 2)
[[ 1.81645675e+00 -2.02871018e-01 -1.03720905e+03]
 [ 1.21520895e-01  1.57952506e+00  6.44470694e+02]
 [ 8.33394824e-06 -1.52971428e-04  1.00000000e+00]]
旋转角度 (度): 5.45646761246657
无人机图像中心点在tif的位置：[ 654.06005013 1586.66676502]
无人机图像中心点在地图上的位置：654.0600501338041,6086.666765018521
无人机图像中心点的经纬度：32.05782459493105, 118.7862248335991
UAV: seu_uav_0103_2/00074.png - TIF: seu_tif_m300/22_0_4500.tif
(1370, 2)
[[ 1.82301257e+00 -2.16703152e-01 -1.03364398e+03]
 [ 1.28793219e-01  1.57890344e+00  6.415712

In [11]:
# from my_pkg.tools import extract_rotation_angle, rotate_point_z
# with open(loc_path, 'w') as loc_file:
#     i = 0
#     for img_uav, img_tif in pairs:
#         if i == 0:
#             print(f"UAV: {img_uav} - TIF: {img_tif}")
#             kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
#             matches,scores = get_matches(matches_path, img_uav, img_tif)
#             print(matches.shape)
#             pts1 = kp1[matches[:,0]]
#             pts2 = kp2[matches[:,1]]
#             H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC)
#             #rotation from H
#             theta = np.arctan2(H[1,0], H[0,0])
#             print(f"theta: {theta}")
#             print(H)
#             i+=1
# import numpy as np
# import cv2
# 
# # 示例 H 矩阵（请替换为你的实际 H 矩阵）
# # H = np.array([
# #     [1.2, 0.3, 100],
# #     [0.1, 1.1, 50],
# #     [0.001, 0.002, 1]
# # ])
# 
# angle = extract_rotation_angle(H)
# print(f"旋转角度 (度): {angle:.2f}")
# 
# point1 = np.array([-0.53, 160.55, 0])
# rotated_point1 = rotate_point_z(point1, -3.5)
# print(f"旋转前的点: {point1}")
# print(f"旋转后的点: {rotated_point1}")

UAV: seu_uav_12-26/00000.png - TIF: seu_tif/2_2000_0.tif
(596, 2)
theta: 0.02056903272280777
[[ 2.46189559e+00  6.87898300e-02 -2.60443082e+02]
 [ 5.06459536e-02  2.69051556e+00 -1.19204901e+03]
 [-5.61151767e-05  6.94698731e-05  1.00000000e+00]]
旋转角度 (度): -0.20
旋转前的点: [ -0.53 160.55   0.  ]
旋转后的点: (array([  9.27233158, 160.28289761,   0.        ]), array([[ 0.9981348 ,  0.06104854,  0.        ],
       [-0.06104854,  0.9981348 ,  0.        ],
       [ 0.        ,  0.        ,  1.        ]]))


# 生成地图轨迹

In [5]:
import cv2
from my_pkg.tools import parse_keyframe_file, draw_keyframe_trajectory
points_traj = []
with open(loc_path, 'r') as loc_file:
    for line in loc_file:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        x_in_map = float(parts[3]) * 0.2  # 调整横坐标
        y_in_map = float(parts[4]) * 0.2  # 调整纵坐标
        print(f"{parts[0]}: {x_in_map}, {y_in_map}")
        points_traj.append((x_in_map, y_in_map))

    # 将点转换为整数坐标（像素坐标）
points_traj = [(int(x), int(y)) for x, y in points_traj]

# plot_traj_tif(
#     map_image_path="/home/lty/data/SEU/seu_resized/seu_resized.tif",
#     loc_file_path=loc_path,
#     output_image_path=output_dir/"traj_map.tif",
#     scale_factor=0.2,
# )

map = cv2.imread("/home/lty/outputs/scene_match_0103_seu_2/gt.png", cv2.IMREAD_COLOR)
keyframe_mapping = parse_keyframe_file("/home/lty/code/ORB_SLAM3_detailed_comments/KeyFrameId.txt")
map_with_traj = draw_keyframe_trajectory(map, points_traj, keyframe_mapping)
cv2.imwrite(output_dir/"traj_map.png", map_with_traj)

seu_uav_0103_2/00000.png: 120.80700927000001, 1219.41385652
seu_uav_0103_2/00073.png: 130.81201002600002, 1217.333353004
seu_uav_0103_2/00074.png: 130.933083858, 1216.632584854
seu_uav_0103_2/00075.png: 131.35034994, 1215.9711842479999
seu_uav_0103_2/00076.png: 132.557658024, 1214.613550158
seu_uav_0103_2/00077.png: 133.13914726200002, 1214.547295328
seu_uav_0103_2/00078.png: 134.192480544, 1215.0638332
seu_uav_0103_2/00079.png: 135.572497936, 1211.982505448
seu_uav_0103_2/00080.png: 135.457590738, 1213.4320622040002
seu_uav_0103_2/00081.png: 136.015080396, 1212.887099642
seu_uav_0103_2/00082.png: 137.092221254, 1212.1941956320002
seu_uav_0103_2/00083.png: 137.766988878, 1212.0905731139999
seu_uav_0103_2/00084.png: 138.364379084, 1212.5312147680002
seu_uav_0103_2/00085.png: 139.3125704, 1212.633448244
seu_uav_0103_2/00086.png: 140.55318501600001, 1212.4084066
seu_uav_0103_2/00087.png: 141.136378116, 1211.711665876
seu_uav_0103_2/00088.png: 142.0534499, 1211.06092727
seu_uav_0103_2/0008

True

slam traj

In [8]:
#slam traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj
slam_traj_path = "/home/lty/outputs/scene_match_0103_seu_2/geoKFrame.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx * 0.2  # 调整横坐标
        y_in_map = Pixely * 0.2
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"traj_map.png", cv2.IMREAD_COLOR)
map_with_traj = draw_slam_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"slam_traj_map.png", map_with_traj)

True

In [11]:
# 打开 HDF5 文件
with h5py.File(features_path, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

    # 示例：读取一个特定图像的特征
    example_image = "seu_uav/DJI_0218.JPG"  # 替换为您的图像名称
    if example_image in f:
        group = f[example_image]
        print(f"\n特定图像 '{example_image}' 的内容:")
        for key in group.keys():
            data = group[key][:]
            print(f"{key}: {data.shape} - {data.dtype}")
    else:
        print(f"图像 '{example_image}' 不存在于 HDF5 文件中。")

文件中的数据集和分组结构：
Group: seu_tif
Group: seu_tif/10_18000_0.tif
Dataset: seu_tif/10_18000_0.tif/descriptors - Shape: (256, 787) - Type: float16
Dataset: seu_tif/10_18000_0.tif/image_size - Shape: (2,) - Type: int64
Dataset: seu_tif/10_18000_0.tif/keypoints - Shape: (787, 2) - Type: float16
Dataset: seu_tif/10_18000_0.tif/scores - Shape: (787,) - Type: float16
Group: seu_tif/11_0_2000.tif
Dataset: seu_tif/11_0_2000.tif/descriptors - Shape: (256, 4096) - Type: float16
Dataset: seu_tif/11_0_2000.tif/image_size - Shape: (2,) - Type: int64
Dataset: seu_tif/11_0_2000.tif/keypoints - Shape: (4096, 2) - Type: float16
Dataset: seu_tif/11_0_2000.tif/scores - Shape: (4096,) - Type: float16
Group: seu_tif/12_2000_2000.tif
Dataset: seu_tif/12_2000_2000.tif/descriptors - Shape: (256, 4096) - Type: float16
Dataset: seu_tif/12_2000_2000.tif/image_size - Shape: (2,) - Type: int64
Dataset: seu_tif/12_2000_2000.tif/keypoints - Shape: (4096, 2) - Type: float16
Dataset: seu_tif/12_2000_2000.tif/scores - Shape: 